# Open Graph Benchmark (OGB)

OGB-backed loaders are folded into the existing **domain categories**
rather than living under a separate ``ogb`` category. They appear
automatically once the ``ogb`` extra is installed:

| Dataset         | Task        | Category   |
|-----------------|-------------|------------|
| ``ogbn-arxiv``  | node_cls    | social     |
| ``ogbn-products`` | node_cls  | finance    |
| ``ogbg-molhiv`` | graph_cls   | biology    |
| ``ogbg-molpcba``| graph_cls   | biology    |
| ``ogbl-collab`` | link_pred   | social     |

Install requirements:

```bash
pip install graphnetz[ogb]          # all five loaders
pip install graphnetz[ogb,chem]     # also installs RDKit, required for molhiv/molpcba
```

## Loading ogbn-arxiv directly

The loader returns a single PyG ``Data`` with ``train_mask`` / ``val_mask``
/ ``test_mask`` populated from OGB's official split, plus ``num_features``
and ``num_classes`` for the trainer dispatcher.

In [ ]:
from graphnetz.datasets.social import ogbn_arxiv

data = ogbn_arxiv('data/ogb')
print(f'Nodes:    {data.num_nodes:,}')
print(f'Edges:    {data.edge_index.size(1):,}')
print(f'Features: {data.num_features}')
print(f'Classes:  {data.num_classes}')
print(f'Train / val / test: {int(data.train_mask.sum()):,} / {int(data.val_mask.sum()):,} / {int(data.test_mask.sum()):,}')


## Single-model training

Any ``nn.Module`` whose forward signature matches ``forward(data)`` works
with ``train_node_classification``. We use the built-in ``GCN`` here.

In [ ]:
import torch
from graphnetz import GCN, plot_history, train_node_classification

torch.manual_seed(0)
model = GCN(data.num_features, 128, data.num_classes)
history = train_node_classification(model, data, epochs=50, lr=0.01)
fig, ax = plot_history(history, title='GCN on ogbn-arxiv')

## Multi-seed benchmark — OGB datasets via their domain categories

Because each OGB dataset lives in a domain category, you reach it
through the matching ``run_benchmark(category, ..., only=[...])`` call.
Filtering to the OGB tasks isolates them from the curated built-ins.

ogbn-arxiv is large enough that 10 seeds × 3 models takes a while; we
use three seeds and 30 epochs for the demo — swap in
``seeds=tuple(range(10))`` and ``epochs=200`` for a publishable run.

In [ ]:
from graphnetz import GAT, GCN, GraphSAGE, run_benchmark

report = run_benchmark(
    'social',
    {'GCN': GCN, 'GAT': GAT, 'GraphSAGE': GraphSAGE},
    task'node_cls',
    only=['ogbn_arxiv'],
    seeds=(0, 1, 2),
    epochs=30,
    root='data/ogb',
)
print(report.summary())

In [ ]:
print(report.pairwise())

### Or sweep alongside the curated built-ins

Drop the ``only=`` filter and ``run_benchmark`` will run every model on
every node-classification task in the ``social`` category — Cora,
CiteSeer, PubMed, the heterophilous suite, **and** ``ogbn_arxiv`` —
yielding a single critical-difference diagram across the lot.

## Graph classification: ogbg-molhiv

``ogbg_molhiv`` returns the raw ``GraphPropPredDataset``. RDKit must be
available — install ``graphnetz[chem]`` (or any environment that already
provides ``rdkit``).

In [ ]:
from graphnetz.datasets.biology import ogbg_molhiv

ds = ogbg_molhiv('data/ogb')
split = ds.get_idx_split()
print(f'Total graphs: {len(ds):,}')
print(f"Train / val / test: {len(split['train']):,} / {len(split['valid']):,} / {len(split['test']):,}")
graph, label = ds[0]
print(f"First graph nodes={graph['num_nodes']}, edges={graph['edge_index'].shape[1]}, label={label.item()}")

## Notes

- OGB tasks are appended to the domain categories lazily — if you
  install the extra after starting Python, restart the kernel before
  importing ``graphnetz``.
- ``run_benchmark`` caches downloaded data under ``root``; the first
  run pays the download cost, subsequent runs are local.
- For publication-grade numbers, push ``seeds`` to 10 and ``epochs`` to
  the dataset-specific defaults registered in ``BENCHMARK_TASKS``
  (e.g. ``BENCHMARK_TASKS['social']['node_cls']`` for ``ogbn_arxiv``).